# Notebook 1: Data Cleaning & Feature Engineering
#
# **Purpose:** Load the excel cleaned CSV, re-create all engineered columns from SQL,
# handle the known data issues, and save a clean file for use in notebooks
# 2 and 3 and Power BI.
#
# **Known issues to handle:**
# - `respiration` was already dropped in Excel before import
# - `vdate` and `discharged` are in yyyymmdd integer format — convert to datetime
# - `lengthofstay` <= 0 are data errors — remove them



In [10]:
import pandas as pd
import numpy as np



# Load the raw data
# Change the path below to wherever the excel cleaned CSV is saved.


In [11]:
df = pd.read_csv(r'hospital_los_excel_cleaned.csv')

print(f"Rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
df.head()



Rows: 100,000
Columns: ['eid', 'vdate', 'rcount', 'gender', 'dialysisrenalendstage', 'asthma', 'irondef', 'pneum', 'substancedependence', 'psychologicaldisordermajor', 'depress', 'psychother', 'fibrosisandother', 'malnutrition', 'hemo', 'hematocrit', 'neutrophils', 'sodium', 'glucose', 'bloodureanitro', 'creatinine', 'bmi', 'pulse', 'secondarydiagnosisnonicd9', 'discharged', 'facid', 'lengthofstay']


,eid,vdate,rcount,gender,dialysisrenalendstage,asthma,irondef,pneum,substancedependence,psychologicaldisordermajor,...,sodium,glucose,bloodureanitro,creatinine,bmi,pulse,secondarydiagnosisnonicd9,discharged,facid,lengthofstay
0,19689,20121028,0,F,0,0,0,0,0,1,...,151.387283,130.012124,10.0,1.028094,28.899694,78,1,20121031,E,3
1,26063,20121208,0,M,0,0,0,0,0,1,...,151.315937,99.959761,10.0,1.273632,27.522360,70,0,20121212,E,4
2,43136,20120122,0,M,0,0,0,0,0,0,...,150.817989,105.888219,70.0,1.030775,25.141062,81,3,20120128,E,6
3,87067,20120418,3,M,0,0,1,0,0,0,...,150.003188,126.215248,14.0,0.858735,27.656839,85,5,20120427,C,9
4,2689,20121201,0,M,0,0,0,0,0,0,...,149.927392,91.217823,30.0,1.017631,29.287791,88,3,20121207,C,6


# ## Step 1: Inspect data types
#
# Before fixing anything, check what pandas inferred for each column.
# Date columns may have imported as integers (yyyymmdd format).



In [12]:
df.dtypes

eid                             int64
vdate                           int64
rcount                          int64
gender                            str
dialysisrenalendstage           int64
asthma                          int64
irondef                         int64
pneum                           int64
substancedependence             int64
psychologicaldisordermajor      int64
depress                         int64
psychother                      int64
fibrosisandother                int64
malnutrition                    int64
hemo                            int64
hematocrit                    float64
neutrophils                   float64
sodium                        float64
glucose                       float64
bloodureanitro                float64
creatinine                    float64
bmi                           float64
pulse                           int64
secondarydiagnosisnonicd9       int64
discharged                      int64
facid                             str
lengthofstay

In [13]:
df.describe()

,eid,vdate,rcount,dialysisrenalendstage,asthma,irondef,pneum,substancedependence,psychologicaldisordermajor,depress,...,neutrophils,sodium,glucose,bloodureanitro,creatinine,bmi,pulse,secondarydiagnosisnonicd9,discharged,lengthofstay
count,100000.000000,1.000000e+05,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,...,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,1.000000e+05,100000.00000
mean,50000.500000,2.012069e+07,1.118210,0.036420,0.035270,0.094940,0.039450,0.063060,0.239040,0.051660,...,10.177455,137.891397,141.963384,14.097185,1.099350,29.805759,73.444720,2.123310,2.012080e+07,4.00103
std,28867.657797,5.935500e+02,1.542958,0.187334,0.184462,0.293134,0.194664,0.243072,0.426499,0.221341,...,5.353131,2.999669,29.992996,12.952454,0.200262,2.003769,11.644555,2.050641,1.151604e+03,2.36031
min,1.000000,2.012010e+07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.100000,124.912632,-1.005927,1.000000,0.219770,21.992683,21.000000,0.000000,2.012010e+07,1.00000
25%,25000.750000,2.012040e+07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,7.700000,135.871062,121.682383,11.000000,0.964720,28.454235,66.000000,1.000000,2.012040e+07,2.00000
50%,50000.500000,2.012070e+07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,9.400000,137.887151,142.088545,12.000000,1.098764,29.807516,73.000000,1.000000,2.012071e+07,4.00000
75%,75000.250000,2.012100e+07,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,11.500000,139.912885,162.180996,14.000000,1.234867,31.156885,81.000000,3.000000,2.012101e+07,6.00000
max,100000.000000,2.013010e+07,5.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,245.900000,151.387283,271.444277,682.500000,2.035202,38.935293,130.000000,10.000000,2.013011e+07,17.00000


# ## Step 2: Fix date columns
#
# `vdate` and `discharged` were saved in yyyymmdd integer format in Excel.
# We convert them to proper datetime objects so pandas can do date arithmetic.



In [14]:
df['vdate']      = pd.to_datetime(df['vdate'].astype(str), format='%Y%m%d')
df['discharged'] = pd.to_datetime(df['discharged'].astype(str), format='%Y%m%d')

print(f"Admission date range: {df['vdate'].min().date()} to {df['vdate'].max().date()}")
print(f"Discharge date range: {df['discharged'].min().date()} to {df['discharged'].max().date()}")



Admission date range: 2012-01-01 to 2013-01-01
Discharge date range: 2012-01-02 to 2013-01-13


# ## Step 3: Validate and clean lengthofstay
#
# LOS must be >= 1. Zero or negative values are data entry errors.
# We remove them and record how many rows were dropped.



In [15]:
invalid_rows = df[df['lengthofstay'] <= 0]
print(f"Rows with LOS <= 0: {len(invalid_rows)}")

df = df[df['lengthofstay'] > 0].copy()
print(f"Rows remaining after removal: {len(df):,}")



Rows with LOS <= 0: 0
Rows remaining after removal: 100,000


# ## Step 4: NULL check across all columns
#
# Check every column for missing values.
# The satisfaction score equivalent here is any lab value with unexpected NULLs.



In [16]:
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]

if len(null_counts) == 0:
    print("No NULL values found in any column.")
else:
    print("Columns with NULLs:")
    print(null_counts)



No NULL values found in any column.


# ## Step 5: Engineer new columns
#
# These match the columns created in SQL. We recreate them here so the
# Python analysis is fully self-contained and does not depend on the SQL database.



# 5a: comorbidity_count — total number of conditions per patient

In [ ]:
comorbidity_cols = [
    'dialysisrenalendstage', 'asthma', 'irondef', 'pneum',
    'substancedependence', 'psychologicaldisordermajor', 'depress',
    'psychother', 'fibrosisandother', 'malnutrition'
]
df['comorbidity_count'] = df[comorbidity_cols].sum(axis=1)

print("Comorbidity count distribution:")
print(df['comorbidity_count'].value_counts().sort_index())



Comorbidity count distribution:
comorbidity_count
0    60140
1    22481
2    11149
3     4076
4     1564
5      476
6       94
7       17
8        3
Name: count, dtype: int64


# 5b: admission_month and admission_year — for seasonal trends

In [ ]:
df['admission_month'] = df['vdate'].dt.month
df['admission_year']  = df['vdate'].dt.year

print(f"Years in dataset: {sorted(df['admission_year'].unique())}")
print(f"Months in dataset: {sorted(df['admission_month'].unique())}")



Years in dataset: [np.int32(2012), np.int32(2013)]
Months in dataset: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]


# 5c: los_category — human-readable LOS bands

In [19]:
def categorise_los(los):
    if los <= 3:
        return '1 - Short (1-3 days)'
    elif los <= 7:
        return '2 - Medium (4-7 days)'
    elif los <= 14:
        return '3 - Long (8-14 days)'
    else:
        return '4 - Extended (15+ days)'

df['los_category'] = df['lengthofstay'].apply(categorise_los)

print("LOS category distribution:")
print(df['los_category'].value_counts().sort_index())



LOS category distribution:
los_category
1 - Short (1-3 days)       46872
2 - Medium (4-7 days)      44563
3 - Long (8-14 days)        8539
4 - Extended (15+ days)       26
Name: count, dtype: int64


# 6d: patient_type — first visit vs. readmitted

In [21]:

df['patient_type'] = df['rcount'].apply(
    lambda x: 'First Visit' if x == 0 else 'Readmitted'
)

print("Patient type split:")
print(df['patient_type'].value_counts())



Patient type split:
patient_type
First Visit    55031
Readmitted     44969
Name: count, dtype: int64


# ## Step 6: Final verification


In [ ]:
print(f"Final row count: {len(df):,}")
print(f"\nColumn list:")
print(df.columns.tolist())
print(f"\nData types:")
print(df.dtypes)



Quick sanity check on key columns

In [ ]:
print("LOS summary:")
print(df['lengthofstay'].describe().round(2))

print("\nrcount_clean summary:")
print(df['rcount_clean'].describe().round(2))

print("\ncomorbidity_count summary:")
print(df['comorbidity_count'].describe().round(2))



# ## Step 7: Save cleaned dataset
#
# This file is used by Notebook 2, Notebook 3, and Power BI.



In [ ]:
df.to_csv(r'data/hospital_los_python_cleaned.csv', index=False)
print("Saved: cleaned_data/hospital_los_python_cleaned.csv")
print(f"Shape: {df.shape}")


Saved: cleaned_data/hospital_los_python_cleaned.csv
Shape: (100000, 32)
